# Main.py

Ini buat Install Library

In [ ]:
!pip install -qU \
    opentelemetry-api==1.38.0 \
    opentelemetry-sdk==1.38.0 \
    opentelemetry-exporter-otlp-proto-common==1.38.0 \
    opentelemetry-proto==1.38.0 \
    langchain-groq \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    pypdf

Ini Nginstall PDF Reader

In [ ]:
!pip install pypdf
!pip install langchain-community pypdf

Ini buat import database

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('Dokumen_Tes_RAG_Rexaldo.pdf')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

print("Database Dora the explorer berhasil dibuat yey >.<")

Yang ini,eksekusi buat RAG

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup API
os.environ["GROQ_API_KEY"] = "RAHASIA GES"

# 2. Panggil LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

# 3. Setup Prompt
system_prompt = (
    "Kamu adalah asisten AI yang sangat cerdas.\n"
    "Gunakan data berikut untuk menjawab pertanyaan karyawan.\n"
    "Jika jawabannya tidak ada di data ini, bilang saja 'Data tidak ditemukan'.\n\n"
    "Konteks:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 4. Fungsi buat ngerapiin teks dari database
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 5. RAG CHAIN MURNI (Tanpa langchain.chains!)
retriever = vectorstore.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Eksekusi
query = "Kalo semisalkan pegawai ada yang telat 20 menit setelah jam kerja,apa yang didapat?"
response = rag_chain.invoke(query)

print(f"Hasil AI: {response}")

Yang ini buat output pertanyaan sama hasil

In [ ]:
pertanyaan_baru = "Gimana cara dapat tunjangan?"
output = rag_chain.invoke(pertanyaan_baru)

print(f"Pertanyaan: {pertanyaan_baru}")
print(f"Jawaban: {output}")